# 1 · Visual Prompt Tuning in code

This notebook builds the model used in every experiment of the talk and checks it against the paper
(Jia et al., *Visual Prompt Tuning*, ECCV 2022, Sec. 3).

**Base model.** ViT-B/16 pretrained with labels on ImageNet-21k, loaded from timm as
`vit_base_patch16_224.orig_in21k`. It is the checkpoint the paper uses. An image of 224 × 224 pixels is cut
into 14 × 14 = 196 patches of 16 × 16 pixels; each patch becomes a 768-dimensional token, a learned
[CLS] token is prepended, and the sequence of 197 tokens passes through 12 Transformer layers.

**VPT.** The base model stays frozen. VPT adds *p* learned vectors ("prompts") to the token sequence:
*VPT-shallow* inserts them once, before layer 1; *VPT-deep* inserts a fresh set before every layer and
discards the previous layer's outputs at those positions. Only the prompts and a linear classification
head are trained.

In [1]:
import sys, json, time, math
from pathlib import Path
ROOT = Path.cwd().parent            # demo/vpt
sys.path.insert(0, str(ROOT))
import numpy as np, torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

torch 2.11.0+cu128 | CUDA: NVIDIA GeForce RTX 5070 Ti Laptop GPU


## The base model

In [2]:
import timm
BACKBONE = "vit_base_patch16_224.orig_in21k"
vit = timm.create_model(BACKBONE, pretrained=True, num_classes=0)   # num_classes=0: no head, outputs the 768-d feature
n_params = sum(p.numel() for p in vit.parameters())
print(f"{BACKBONE}: {n_params:,} parameters ({n_params / 1e6:.1f}M; the paper reports 85.8M)")
print("patch size", vit.patch_embed.patch_size, "| width d =", vit.embed_dim, "| layers N =", len(vit.blocks))
print("input normalisation: mean", vit.pretrained_cfg["mean"], "std", vit.pretrained_cfg["std"])

vit_base_patch16_224.orig_in21k: 85,798,656 parameters (85.8M; the paper reports 85.8M)
patch size (16, 16) | width d = 768 | layers N = 12
input normalisation: mean (0.5, 0.5, 0.5) std (0.5, 0.5, 0.5)


## The VPT model

One class covers the three settings of the talk: `num_prompts=0` is the linear probe, `deep=False` is
VPT-shallow, `deep=True` is VPT-deep. The same class is saved as `vpt_model.py` and imported by the
other notebooks. Prompts are initialised with Xavier-uniform as in the official code, and get a dropout
of 0.1 during training (paper, Appendix A).

In [3]:
import torch.nn as nn

class VPTViT(nn.Module):
    def __init__(self, num_classes, num_prompts=50, deep=True, prompt_dropout=0.1, vit=None):
        super().__init__()
        self.vit = vit if vit is not None else timm.create_model(BACKBONE, pretrained=True, num_classes=0)
        for p in self.vit.parameters():
            p.requires_grad = False                       # the base model is frozen
        d, n_layers = self.vit.embed_dim, len(self.vit.blocks)
        self.num_prompts, self.deep = num_prompts, deep
        self.prompt_dropout = nn.Dropout(prompt_dropout)
        if num_prompts > 0:
            patch = self.vit.patch_embed.patch_size[0]
            bound = math.sqrt(6.0 / (3 * patch * patch + d))      # Xavier-uniform, as in the official code
            self.prompts = nn.Parameter(torch.empty(n_layers if deep else 1, num_prompts, d).uniform_(-bound, bound))
        else:
            self.register_parameter("prompts", None)
        self.head = nn.Linear(d, num_classes)

    def train(self, mode=True):
        super().train(mode)
        self.vit.eval()                                   # no dropout or drop-path inside the frozen model
        return self

    def forward(self, x):
        v, p = self.vit, self.num_prompts
        x = v.norm_pre(v._pos_embed(v.patch_embed(x)))    # [CLS] + 196 patch tokens, with position embeddings
        for i, blk in enumerate(v.blocks):
            if p and (i == 0 or self.deep):
                P = self.prompt_dropout(self.prompts[i if self.deep else 0]).expand(x.shape[0], -1, -1)
                rest = x[:, 1:] if i == 0 else x[:, 1 + p:]   # layer 1: insert; deeper: replace the old prompt outputs
                x = torch.cat([x[:, :1], P, rest], dim=1)
            x = blk(x)
        return self.head(v.norm(x)[:, 0])                 # classify from the final [CLS] token

## Parameter counts against the paper

Sec. 3.2 of the paper: with ViT-B (d = 768) and 50 prompts, VPT-shallow adds p·d = 0.038M parameters and
VPT-deep adds N·p·d = 0.46M, i.e. 0.04% and 0.53% of the backbone. The head comes on top (here for the
7 classes of DermaMNIST).

In [4]:
rows = []
for name, kw in [("linear probe", dict(num_prompts=0)), ("VPT-shallow, p=50", dict(num_prompts=50, deep=False)), ("VPT-deep, p=50", dict(num_prompts=50, deep=True))]:
    m = VPTViT(7, vit=vit, **kw)
    prompts = m.prompts.numel() if m.prompts is not None else 0
    head = sum(p.numel() for p in m.head.parameters())
    trainable = sum(p.numel() for p in m.parameters() if p.requires_grad)
    rows.append((name, prompts, head, trainable, 100 * trainable / n_params))
print(f"{'':20s}{'prompts':>10s}{'head':>8s}{'trainable':>12s}{'% of backbone':>15s}")
for r in rows:
    print(f"{r[0]:20s}{r[1]:>10,}{r[2]:>8,}{r[3]:>12,}{r[4]:>14.3f}%")

                       prompts    head   trainable  % of backbone
linear probe                 0   5,383       5,383         0.006%
VPT-shallow, p=50       38,400   5,383      43,783         0.051%
VPT-deep, p=50         460,800   5,383     466,183         0.543%


## The token sequence really grows

A forward hook on the first and last layer shows the sequence length each one receives:
197 tokens without prompts, 1 + 50 + 196 = 247 with them.

In [5]:
x = torch.randn(2, 3, 224, 224)
for name, kw in [("linear probe", dict(num_prompts=0)), ("VPT-deep", dict(num_prompts=50, deep=True))]:
    m = VPTViT(7, vit=vit, **kw).eval()
    seen = {}
    hooks = [m.vit.blocks[i].register_forward_pre_hook(lambda mod, inp, i=i: seen.__setitem__(i, tuple(inp[0].shape))) for i in (0, 11)]
    with torch.no_grad():
        out = m(x)
    for h in hooks:
        h.remove()
    print(f"{name:13s} layer 1 input {seen[0]}, layer 12 input {seen[11]}, logits {tuple(out.shape)}")

linear probe  layer 1 input (2, 197, 768), layer 12 input (2, 197, 768), logits (2, 7)


VPT-deep      layer 1 input (2, 247, 768), layer 12 input (2, 247, 768), logits (2, 7)

## Only the prompts and the head learn

One training step with a random batch: the frozen base model receives no gradient at all, while the
prompts of every layer do. (Gradients still flow *through* the frozen layers to reach the prompts,
which is why VPT's training memory is close to full fine-tuning's; see notebook 4.)

In [6]:
m = VPTViT(7, vit=vit, num_prompts=50, deep=True).train()
loss = nn.functional.cross_entropy(m(torch.randn(4, 3, 224, 224)), torch.randint(0, 7, (4,)))
loss.backward()
frozen_with_grad = sum(1 for p in m.vit.parameters() if p.grad is not None)
per_layer = m.prompts.grad.abs().sum(dim=(1, 2))
print("backbone tensors that received a gradient:", frozen_with_grad)
print("prompt gradient magnitude per layer:", [f"{g:.2e}" for g in per_layer.tolist()])
print("head gradient norm:", f"{m.head.weight.grad.norm():.3f}")

backbone tensors that received a gradient: 0
prompt gradient magnitude per layer: ['1.15e-01', '3.30e-01', '7.74e-01', '1.27e+00', '1.47e+00', '1.67e+00', '1.26e+00', '1.31e+00', '1.29e+00', '3.43e+00', '1.14e+01', '2.91e+01']
head gradient norm: 0.736
